In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib.pyplot as plt
import scipy.optimize as opt
import math
import pandas as pd

# Assignment 1 — Introduction: Understanding a Model

## The Model: Fitts' Law

To understand the different facets of an explanatory model, let's consider a simple one: a mathematical model of response times for motor movements, known as Fitts' law. Fitts' law is an equation that predicts the time required to move a hand (or a cursor, or a pen) to a target area that has width $W$ and is located at a distance $D$ from the starting position of the hand. According to Fitts' law, the time $T$ is related to width $W$ and distance $D$ by the following equation:

\begin{equation}
    T = a + b \log_2 \left( \frac{2D}{W} \right)
\end{equation}

with $\log_2(2D/W)$ indexing the difficulty: targets that are farther away (large $D$) or smaller (small $W$) are harder to hit, and therefore take longer.

In [ ]:
# Fitts' law
def fitts(d, w, a=0.81, b=1.12):
    """Fitts' law"""
    return a + b * (math.log2( 2 * d / w))

## Inside a Model: Fit, Features, and Free Parameters

If we peek inside a model (_any_ model), we can find some common elements.

**Output**: Any model must have an output. In the case of Fitts' law, the output is the movement time $T$. This output might or might not reflect the data; the degree to which the model's output matches the data is called the model's **fit**.

**Features**: Each model contains certain quantities that capture specific aspects of the outside world and environment. In Fitts' law, these are **W** and **D**, the width and distance of the target. In choosing the appropriate features, the designer of a model implicitly defines the level of abstraction and the degree of simplification they want to impose on the world. With the level of abstraction fixed, features are in principle _measurable properties of the world_, independent of the model (e.g., you can measure the distance to the target without knowing Fitts' law).

**Free parameters**: The equation also contains **a** and **b**. Unlike **W** and **D**, these do not correspond to anything directly measurable in the world. Instead, they only exist inside the equation, and their values are chosen to make the model fit. Because Fitts' law is an explanatory model, its parameters carry meaning:

- **a** is the intercept. It is the smallest time it takes to initiate a movement on the given device. Hence, no predicted time can ever be smaller than **a**.
- **b** is the slope. It measures the time cost of increased difficulty: a larger **b** means the participant is _more_ penalized by difficult targets. The participant's movement time grows faster as targets get farther away or smaller.

## Fitting a Model: The Loss Function

Now that we have identified the features and free parameters, how do we know whether our model is any good? We need to find the parameter values that make the model's predictions as close as possible to the data. The function that measures the mismatch between actual and predicted data is the **loss function**.

Suppose we ran an experiment, varying the distance $D$ to a target of width $W$, and recorded the movement time on each trial. Here is example data for our first participant:

In [ ]:
data1 = {"W" : [100, 150, 220, 110, 40],
        "D" : [300, 50, 100, 200, 250],
       "time" : [5.23, 1.97, 1.66, 4.04, 5.14]
}

df1 = pd.DataFrame(data1)

df1

A convenient loss is the **sum of squared errors** between the model's predictions $Y'$ and the observed times $Y$:

$$L = \sum_{y \in Y} (y-y')^2$$

Fitting a model is, therefore, the process of finding the values of parameters $a$ and $b$ that minimize the loss function's output **L**.

In [ ]:
def loss_fitts(df, a, b):
    """Loss function for Fitts' law"""
    Ypred = [fitts(x[0], x[1], a=a, b=b) for x in zip(df.D, df.W)]
    Yobs = df.time
    return np.sum((Ypred - Yobs)**2)

## Fitting by Direct Calculation — Linear Regression

Sometimes we are lucky and a formula gives the best parameters directly. Fitts' law is such a case. If you treat $x = \log_2(2D/W)$ as a single predictor, the equation $T = a + b\,x$ is an ordinary straight line, so the best-fitting $a$ and $b$ come from the regression formula $(X^TX)^{-1} X^T Y$.

In [ ]:
def concat_fitts(df):
    # Combine W and D into the single predictor x = log2(2D / W)
    x = np.log2(2 * df.D / df.W)
    X = np.column_stack([np.ones(len(df)), np.asarray(x)])
    Y = df.time.values.reshape(-1, 1)
    return x, X, Y


def lin_reg_fitts(X, Y):
    a_star, b_star = (la.inv(X.T.dot(X)).dot(X.T).dot(Y)).ravel()
    print("a = %.4f, b = %.4f" % (a_star, b_star))
    return a_star, b_star

x1, X1, Y1 = concat_fitts(df1)
a_star_1, b_star_1 = lin_reg_fitts(X1, Y1)
print("loss = %.4f" % loss_fitts(df1, a_star_1, b_star_1))

In [ ]:
def plot_fitts_lin_reg(x, Y, a_star, b_star):
    plt.plot(x, Y, "o")
    xx = np.linspace(min(x), max(x))
    yy = a_star + b_star*xx
    plt.plot(xx, yy, "--")
    plt.xlabel("$X = log_2(2D/W$)")
    plt.ylabel("$Y$ = Time $T$")
    plt.title("Fitts' Law as Linear Regression")
    plt.text(x=1, y=2, s="$y$ = %.3f + %.3f $log (2D/W)$" % (a_star, b_star))
    plt.show()


plot_fitts_lin_reg(x1, Y1, a_star_1, b_star_1)

## Grid Search

What if no formula is available? The brute-force option is to try many combinations of $a$ and $b$, such as every value from 0.5 to 2.5 in steps of 0.04, and keep the one with the smallest loss. This is **grid search**. In the plot below, color shows the loss (darker is smaller, i.e., a better fit), and the white "+" marks the linear-regression solution in parameter space.

In [ ]:
def grid_search_fitts(data, a_star, b_star, low_b=0.5, up_b=2.5):
    A = np.linspace(low_b, up_b)
    B = np.linspace(low_b, up_b)
    grid = np.zeros((A.size, B.size))
    for i, a in enumerate(A):
        for j, b in enumerate(B):
            grid[i,j] = loss_fitts(data, a, b)
            
    
    fig = plt.figure(figsize=(5, 3.2))
    
    ax = fig.add_subplot(111)
    ax.set_title('Loss Function Across Parameters')
    X,Y = np.meshgrid(A, B)
    plt.pcolormesh(A, B, np.sqrt(grid), shading="auto")
    plt.plot(b_star, a_star, "+", color="white", markersize=14)
    ax.set_aspect('equal')
    ax.set_xlabel("$b$")
    ax.set_ylabel("$a$")
    
    cax = fig.add_axes([0.12, 0.1, 0.78, 0.8])
    cax.get_xaxis().set_visible(False)
    cax.get_yaxis().set_visible(False)
    cax.patch.set_alpha(0)
    cax.set_frame_on(False)
    plt.colorbar(orientation='vertical', label="$\\sqrt{\\mathrm{loss}}$")
    
    plt.show()

grid_search_fitts(df1, a_star_1, b_star_1)

However, this brute-force grid-search approach is rarely used in practice, as sampling the whole parameter space is often infeasible — especially as models become more complex and take longer to run. For example, the plot in the figure above was generated by examining 2,500 combinations of $a$ and $b$ values; such a sample might not always be feasible. Furthermore, grid search requires setting a predefined sampling grid that discretizes the possible values of $a$ and $b$. For example, the grid search examined cases in which $a=1.44$ and $a=1.45$, but never examined the case in which $a = 1.4438$; such a value would, in fact, be invisible to the method. For all of these reasons, it is common to use special techniques called _optimization algorithms_ instead of grid searches.

### Optimization

Optimization algorithms capitalize on the fact that, in most models, similar parameter values produce similar results in terms of the model's loss function. In Fitts' law, for example, changing the value of $a$ from 1.44 to 1.45 does not produce appreciable changes in the loss function, no matter what the value of $b$ is. Furthermore, the direction of the changes in the loss function is usually consistent: if changing $a$ from 1.44 to 1.45 increases the loss function, then a further change of $a$ to 1.46 would likely result in an even larger loss value. In other words, the surface of the loss function over the two parameters' values is smooth. And smooth functions can be explored fairly easily by finding out the direction in which the parameters can be changed to reduce the loss function. This is exactly what optimization algorithms do: they start with an initial guess for the model parameters and modify them iteratively in the direction that reduces the loss function, until a minimum value is found (for this reason, these algorithms are also called _minimization_ algorithms). Optimization algorithms explore only a small portion of the parameter space, but they converge quickly on the correct solution. The figure below depicts the points (in white) explored by one such method, the Nelder-Mead algorithm, to find the values of $a$ and $b$ that minimize the loss function of Fitts' law, starting at $a=1$, $b=1$ and terminating at the same values that were identified by linear regression.

In [ ]:
def optimize_fitts(data, low=0.5, up=2.5):
    path = []

    def vloss(vec):
        path.append(vec.copy())
        return loss_fitts(data, vec[0], vec[1])

    res = opt.minimize(vloss, np.array([1.0, 1.0]), method="Nelder-Mead")

    A = np.linspace(low, up)
    B = np.linspace(low, up)
    grid = np.array([[loss_fitts(data, a, b) for b in B] for a in A])

    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.set_title("Optimization path")
    plt.pcolormesh(B, A, np.sqrt(grid), shading="auto")
    plt.plot([p[1] for p in path], [p[0] for p in path], "-o", color="white", markersize=3)
    plt.plot(path[0][1], path[0][0], "o", color="red")
    plt.text(path[0][1], path[0][0] - 0.1, s="Start", color="red")
    plt.plot(path[-1][1], path[-1][0], "o", color="cyan")
    plt.text(path[-1][1] + 0.05, path[-1][0], s="End", color="cyan")
    ax.set_aspect("equal")
    ax.set_xlabel("$b$")
    ax.set_ylabel("$a$")
    plt.colorbar(orientation="vertical", label="$\\sqrt{\\mathrm{loss}}$")
    ax.set_xlim(low, up)
    ax.set_ylim(low, up)
    plt.show()


optimize_fitts(df1)

## A Note on the Exercises!

For the **coding exercises**, you can usually reuse functions from above, or make variants based on them.

The number of points awarded gives you an indication of the depth expected.

## Comparing Two Participants

We managed to recruit a second participant for the same cursor-pointing task. Their data is below. It is now your turn to analyze it and compare the two participants.

In [ ]:
## Participant 2

data2 = {"W" : [100, 150, 220, 110, 40],
        "D" : [300, 50, 100, 200, 250],
       "time" : [4.43, 2.27, 2.14, 3.8, 5.06]
}
df2 = pd.DataFrame(data2)
df2

### Exercise 1a: Fit Fitts' Law for Participant 2 (2 pts)

Fit Fitts' law to participant 2 exactly as we did for participant 1: use the **linear-regression solution** to obtain $a$ and $b$, produce the **regression-line plot** and the **optimization-path plot**, and compute the **loss**.

In [ ]:
# YOUR CODE HERE

### Exercise 1b: Compare Participants (3 pts)

Now, compare the results of the original participant 1 with those of participant 2.

Plot both participants' data and both fitted regression lines in a **single figure** for comparison (complete the cell below).

Then, interpret the comparison:
- Did the models fit the data well enough for a reliable interpretation of the parameters? Why? Why not?
- What do the fitted parameters say about the performance of the respective participants? How do they differ?

In [ ]:
# Participant 1
plt.plot(x1, Y1, "o", label="participant 1 (data)")
xx1 = np.linspace(min(x1), max(x1))
plt.plot(xx1, a_star_1 + b_star_1 * xx1, "--", label="participant 1 (fit)")

# YOUR CODE HERE: add the plot for participant 2

*Answer (1b):*

_Write your interpretation here._

## Welford's Two-Factor Variant

Fitts' law uses a fixed relationship between distance and width through $2D/W$, with one shared parameter $b$. While this simple model works extremely well, many extensions have been proposed over the years. One of them, **Welford's two-factor variant**, uses the same two features, but gives each its own parameter, allowing width and distance to contribute to movement time independently:

$$T = a + b_1 \log_2 \left( D \right) + b_2 \log_2 \left( W \right)$$

Here, $b_1$ captures the impact of the distance to the target on movement time, while $b_2$ captures the impact of the width of the target. The parameter $a$ remains the intercept.

### Exercise 2a: Implement Welford's Model (2 pts)

Working with **participant 1 only**, implement Welford's variant in the `welfords` function below, then run the analysis. The remaining helper functions are provided in the next cell.

- complete `welfords` (the cell after the helpers);
- fit the parameters with `lin_reg_welfords` and report the **loss** with `loss_welford`;
- plot the optimization path with `optimize_welford`.

In [ ]:
# Provided helper functions for Welford's variant
def concat_welfords(df):
    X = np.column_stack([np.ones(len(df)), np.log2(df.D.values), np.log2(df.W.values)])
    Y = df.time.values.reshape(-1, 1)
    return X, Y

def lin_reg_welfords(X, Y):
    a_star, b1_star, b2_star = (la.inv(X.T @ X) @ (X.T @ Y)).ravel()
    print(f"a = {a_star:.3f}, b1 = {b1_star:.3f}, b2 = {b2_star:.3f}")
    return a_star, b1_star, b2_star

def loss_welford(df, a, b1, b2):
    """Sum-of-squared-errors loss for Welford's variant."""
    Ypred = [welfords(x[0], x[1], a=a, b1=b1, b2=b2) for x in zip(df.D, df.W)]
    return np.sum((Ypred - df.time) ** 2)

def optimize_welford(df):
    path = []

    def vloss(vec):
        path.append(vec.copy())
        return loss_welford(df, vec[0], vec[1], vec[2])

    opt.minimize(vloss, np.array([1.0, 1.0, -1.0]), method="Nelder-Mead")
    p = np.array(path)
    fixed_a = p[-1, 0]

    B1 = np.linspace(p[:, 1].min() - 0.3, p[:, 1].max() + 0.3, 60)
    B2 = np.linspace(p[:, 2].min() - 0.3, p[:, 2].max() + 0.3, 60)
    G = np.array([[loss_welford(df, fixed_a, b1, b2) for b1 in B1] for b2 in B2])

    fig, ax = plt.subplots(figsize=(5.5, 4))
    plt.pcolormesh(B1, B2, np.sqrt(G), shading="auto")
    plt.colorbar(label="$\\sqrt{\\mathrm{loss}}$")
    plt.plot(p[:, 1], p[:, 2], "-o", color="white")
    plt.plot(p[-1, 1], p[-1, 2], "*", color="cyan")
    ax.set_xlabel("$b_1$ (distance)")
    ax.set_ylabel("$b_2$ (width)")
    ax.set_title("Welford optimisation path ($a$ fixed at %.2f)" % fixed_a)
    plt.show()

In [ ]:
def welfords(d, w, a=0.0, b1=0.0, b2=0.0):
    """Welford's two-factor variant of Fitts' law."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: fit participant 1 with lin_reg_welfords, report loss_welford, plot optimize_welford

### Exercise 2b: Evaluate Welford's Variant (3 pts)

Interpret participant 1's Welford fit. What do the three fitted values $a$, $b_1$ and $b_2$ mean? How do they compare to the parameters derived from Fitts' law, and how does the interpretation differ between the two models?

*Answer (2b):*

_Write your interpretation here._

### Exercise 2c: Which Model Fits the Data Better? (2 pts)

Compare participant 1's loss under Fitts' law with the loss under Welford's variant. What do you find? Which model fits the data better? Do you think that the model with the lower loss is preferable to the alternative? Why? Why not?

*Answer (2c):*

_Write your interpretation here._

### Exercise 2d: Which Model/Variant Would You Choose? (2 pts)

Based on what you now know and understand about the two models, which one would you prefer to use if you had to model **a completely new** task involving movements towards targets of differing sizes and distances? Briefly explain why you prefer that model.

*Answer (2d):*

_Write your interpretation here._

## Whose Model Is It Anyway?

We will now use data from two different, very much fictional tasks, assigned to participants 3 and 4. The data sets for both participants were synthesized: one based on Fitts' law, the other on Welford's variant (both with added noise). You do not know which is which.

In [ ]:
# Mystery participant 3
data3 = {"W": [100, 150, 220, 110, 40, 150, 200, 160, 50, 100, 100, 80],
         "D": [300, 50, 100, 200, 250, 150, 100, 300, 200, 250, 50, 150],
         "time": [5.38, 2.93, 3.45, 5.09, 5.15, 4.16, 2.97, 4.93, 5.37, 4.66, 3.01, 4.26]}
df3 = pd.DataFrame(data3)

# Mystery participant 4
data4 = {"W": [100, 150, 220, 110, 40, 150, 200, 160, 50, 100, 100, 80],
         "D": [300, 50, 100, 200, 250, 150, 100, 300, 200, 250, 50, 150],
         "time": [5.07, 2.06, 2.22, 3.31, 5.5, 3.82, 2.64, 4.28, 5.08, 4.37, 2.67, 4.62]}
df4 = pd.DataFrame(data4)

### Exercise 3a: Reveal the Model Citizens! (2 pts)

Work out which participant was generated from which model!

In [ ]:
# YOUR CODE HERE: analyse participants 3 and 4

### Exercise 3b: Explain Their Identities (4 pts)

Which participant came from which model? How did you decide this? Explain why and how your method in _3a_ can actually tell the two models apart.

*Answer (3b):*

_Write your answer here._